# MPRF — Calibration (Brier / ECE)
Placer d'abord les 4 fichiers de données dans `/content` (mêmes noms que dans la cellule précédente), puis exécuter la cellule ci-dessous. Sortie : `mprf_calibration_raw.csv`.

In [ ]:
# ============================================================
# MPRF — Calibration (R1.8) : Brier, ECE, biais moyen, recalibration de Platt
#  - Probabilité = moyenne des probabilités de classe 1 des arbres (même score que pour l'AUPRC)
#  - Méthodes : CSRF, OBP, IBEP, MPRF (alpha=0.6, k=10), 5 seeds, mêmes plis/seeds que cellD
#  - Brier et ECE calculés sur les prédictions hors-pli regroupées (10 plis) de chaque seed ;
#    ± = écart-type entre seeds
#  - "+Platt" : régression logistique ajustée sur la partition de validation (jamais sur le pli de test)
# Aucun fichier du papier n'est modifié. Google Colab — copier-coller directement
# ============================================================
import os, time
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score, brier_score_loss
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from scipy.stats import wilcoxon
import warnings
warnings.filterwarnings("ignore")

# ┌──────────────────────── PARAMÈTRES ────────────────────────┐
DATASETS = ["cervical", "hepatitis", "pima", "mammography"]   # retirer ceux que tu ne veux pas relancer
SEEDS    = [0, 1, 2, 3, 4]          # seed 0 = exactement le protocole du papier
ALPHAS   = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
KS       = [5, 10, 15, 20]
ALPHA0, K0 = 0.6, 10                # configuration du papier
N_TREES, N_SPLITS = 50, 10
NESTED, NESTED_SEEDS, INNER_SPLITS = True, [0], 3   # alpha choisi par CV interne (3 plis) sur le train
N_BOOT   = 2000
OUT_DIR  = "."                      # les CSV sont écrits ici
FILES = {                           # mêmes fichiers que dans ton notebook
    "cervical":    "1. risk_factors_cervical_cancer.csv",
    "hepatitis":   "2. HepatitisCdata.csv",
    "pima":        "3. diabetes.csv",
    "mammography": "/content/phpn1jVwe.arff",
}
# └─────────────────────────────────────────────────────────────┘


# ┌──────────────────────── PARAMÈTRES ────────────────────────┐
DATASETS = ["cervical", "hepatitis", "pima", "mammography"]
SEEDS    = [0, 1, 2, 3, 4]
ALPHA0, K0 = 0.6, 10
N_TREES, N_SPLITS, N_BINS = 50, 10, 10
OUT_DIR  = "."
FILES = {
    "cervical":    "1. risk_factors_cervical_cancer.csv",
    "hepatitis":   "2. HepatitisCdata.csv",
    "pima":        "3. diabetes.csv",
    "mammography": "/content/phpn1jVwe.arff",
}
# └─────────────────────────────────────────────────────────────┘

# ── Chargement (identique à ton notebook) ───────────────────
def load(name):
    path = FILES[name]
    if not os.path.exists(path):
        print(f"⚠️  {name}: fichier introuvable ({path}) → ignoré"); return None
    if name == "cervical":
        df = pd.read_csv(path); df.replace("?", np.nan, inplace=True)
        df = df.apply(pd.to_numeric, errors="coerce"); df.fillna(df.median(numeric_only=True), inplace=True)
        feats = [c for c in df.columns if c != "Biopsy"]
        return df[feats].values, df["Biopsy"].values.astype(int)
    if name == "hepatitis":
        df = pd.read_csv(path); df["Category"] = df["Category"].astype(str).str.strip()
        df["Category"] = df["Category"].map(lambda x: 0 if "Blood Donor" in x else (1 if "Hepatitis" in x else np.nan))
        df = df.dropna(subset=["Category"])
        feats = [c for c in df.columns if c not in ["Category", "Unnamed: 0"]]
        d = df[feats].copy()
        for col in d.columns:
            if d[col].dtype == object:
                le = LabelEncoder(); m = d[col].notna()
                d.loc[m, col] = le.fit_transform(d.loc[m, col].astype(str)); d[col] = pd.to_numeric(d[col], errors="coerce")
        d = d.apply(pd.to_numeric, errors="coerce"); d.fillna(d.median(numeric_only=True), inplace=True)
        return d.values, df["Category"].values.astype(int)
    if name == "pima":
        df = pd.read_csv(path); df.fillna(df.median(numeric_only=True), inplace=True)
        return df[[c for c in df.columns if c != "Outcome"]].values, df["Outcome"].values.astype(int)
    if name == "mammography":
        rows, data = [], False
        for line in open(path):
            line = line.strip()
            if line.lower() == "@data": data = True; continue
            if data and line:
                v = line.split(","); rows.append([float(t) for t in v[:6]] + [1 if v[6].strip().strip("'") == "1" else 0])
        a = np.array(rows); return a[:, :6], a[:, 6].astype(int)

# ── Métriques / score (mêmes formules que le notebook) ──────
def counts(y, p):
    tp = int(((y == 1) & (p == 1)).sum()); fp = int(((y == 0) & (p == 1)).sum())
    fn = int(((y == 1) & (p == 0)).sum()); tn = int(((y == 0) & (p == 0)).sum())
    return tp, fp, fn, tn

def f1_gm(tp, fp, fn, tn):
    den = 2 * tp + fp + fn
    f1 = 2 * tp / den if den > 0 else 0.0
    r1 = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    r0 = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    return f1, np.sqrt(r1 * r0)

def metrics(y, p, proba):
    tp, fp, fn, tn = counts(y, p); f1, gm = f1_gm(tp, fp, fn, tn)
    return dict(f1_min=f1, gmean=gm,
                precision=tp / (tp + fp) if (tp + fp) > 0 else 0.0,
                sens=tp / (tp + fn) if (tp + fn) > 0 else 0.0,
                spec=tn / (tn + fp) if (tn + fp) > 0 else 0.0,
                fp=fp, fn=fn, auprc=average_precision_score(y, proba))

def score_vec(yv, pred_mat, alpha):          # pred_mat : (candidats, n_val) booléens
    pos = (yv == 1); neg = ~pos
    tp = (pred_mat & pos).sum(1); fp = (pred_mat & neg).sum(1)
    fn = (~pred_mat & pos).sum(1); tn = (~pred_mat & neg).sum(1)
    den = 2 * tp + fp + fn
    f1 = np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0)
    r1 = np.where(tp + fn > 0, tp / np.maximum(tp + fn, 1), 0.0)
    r0 = np.where(tn + fp > 0, tn / np.maximum(tn + fp, 1), 0.0)
    return alpha * f1 + (1 - alpha) * np.sqrt(r1 * r0)

# ── Sélection d'arbres (P = matrice (arbres, instances) des prédictions 0/1) ──
def MPRF(Pv, yv, k, alpha, init_seed, init="random"):
    n = Pv.shape[0]
    if init == "random":
        i0 = np.random.RandomState(init_seed).randint(0, n)
    else:                                         # "best" : meilleur arbre seul sur la validation
        i0 = int(np.argmax(score_vec(yv, Pv.astype(bool), alpha)))
    sel = [i0]; rem = [i for i in range(n) if i != i0]
    votes = Pv[i0].astype(float).copy()
    best = score_vec(yv, (votes >= 0.5)[None, :], alpha)[0]
    while len(sel) < k and rem:
        V = votes[None, :] + Pv[rem]
        sc = score_vec(yv, V >= (len(sel) + 1) / 2, alpha)
        j = int(np.argmax(sc))                    # 1er maximum = même règle que la boucle d'origine
        if sc[j] >= best:                         # plateau accepté
            votes = V[j]; sel.append(rem.pop(j))
            if sc[j] > best: best = sc[j]
        else:
            break
    return sel

def OBP(Pv, yv, max_size):
    n = Pv.shape[0]
    sc = [f1_gm(*counts(yv, Pv[i]))[0] for i in range(n)]
    order = np.argsort(sc)[::-1]; sel, best, votes = [], -np.inf, np.zeros(Pv.shape[1])
    for idx in order[:max_size]:
        v = votes + Pv[idx]; s = f1_gm(*counts(yv, (v >= (len(sel) + 1) / 2).astype(int)))[0]
        if s >= best: sel.append(idx); votes = v; best = s
    return sel if sel else [order[0]]

def IBEP(Pv, yv, max_size):
    n = Pv.shape[0]; err = (Pv != yv).astype(float); w = np.ones(len(yv)) / len(yv); sel = []
    for _ in range(max_size):
        b = int(np.argmin(err.dot(w)))
        if b in sel: break
        sel.append(b); w *= np.exp(err[b]); w /= w.sum()
    return sel if sel else [0]

def vote(P, sel):  return (P[sel].sum(0) >= len(sel) / 2).astype(int)

def forest(Xa, ya, seed, balanced):
    return RandomForestClassifier(n_estimators=N_TREES, class_weight="balanced" if balanced else None,
                                  random_state=42 + seed, n_jobs=-1).fit(Xa, ya)

def tree_mats(rf, Xv, Xt):
    Pv = np.array([t.predict(Xv) for t in rf.estimators_])
    Pt = np.array([t.predict(Xt) for t in rf.estimators_])
    Qt = np.array([t.predict_proba(Xt)[:, 1] for t in rf.estimators_])
    return Pv, Pt, Qt

# ── Calibration ─────────────────────────────────────────────
def ece_score(y, p, n_bins=N_BINS, strategy="uniform"):
    y = np.asarray(y, float); p = np.asarray(p, float)
    if strategy == "uniform":
        b = np.minimum((p * n_bins).astype(int), n_bins - 1)
    else:                                         # classes de même effectif
        b = np.minimum((np.argsort(np.argsort(p, kind="stable"), kind="stable") * n_bins // len(p)), n_bins - 1)
    e = 0.0
    for i in range(n_bins):
        m = b == i
        if m.any(): e += m.mean() * abs(y[m].mean() - p[m].mean())
    return e

def platt(sv, yv, st):
    """Régression logistique sur le score brut (validation) puis application au test."""
    if len(np.unique(yv)) < 2: return st
    lr = LogisticRegression(C=1e6, max_iter=1000).fit(np.asarray(sv).reshape(-1, 1), yv)
    return lr.predict_proba(np.asarray(st).reshape(-1, 1))[:, 1]

METHODS = ["CSRF", "OBP", "IBEP", MAIN := f"MPRF a={ALPHA0:.1f} k={K0}"]

def run_cal(name, X, y):
    store = {}                                    # (seed, method) -> listes y, p ; (seed, method, fold) -> Brier
    fold_brier = []
    skf = StratifiedKFold(N_SPLITS, shuffle=True, random_state=42)
    for seed in SEEDS:
        for fold, (tr, te) in enumerate(skf.split(X, y)):
            Xtr, Xte, ytr, yte = X[tr], X[te], y[tr], y[te]
            vi = np.random.RandomState(fold + 1000 * seed).choice(len(ytr), int(len(tr) * 0.2), replace=False)
            sub = np.setdiff1d(np.arange(len(ytr)), vi)
            Xv, yv, Xs, ys = Xtr[vi], ytr[vi], Xtr[sub], ytr[sub]
            iseed = 100 + fold + 1000 * seed
            rf = forest(Xs, ys, seed, True)
            Pv, Pt, Qt = tree_mats(rf, Xv, Xte)
            Qv = np.array([t.predict_proba(Xv)[:, 1] for t in rf.estimators_])
            sels = {"CSRF": list(range(N_TREES)), "OBP": OBP(Pv, yv, N_TREES), "IBEP": IBEP(Pv, yv, N_TREES),
                    MAIN: MPRF(Pv, yv, K0, ALPHA0, iseed)}
            for m, sel in sels.items():
                pt, pv = Qt[sel].mean(0), Qv[sel].mean(0)
                for tag, p in ((m, pt), (m + " +Platt", platt(pv, yv, pt))):
                    store.setdefault((seed, tag), ([], []))
                    store[(seed, tag)][0].extend(yte.tolist()); store[(seed, tag)][1].extend(p.tolist())
                    fold_brier.append(dict(dataset=name, seed=seed, fold=fold, method=tag, brier=brier_score_loss(yte, p)))
        print(f"   {name}: seed {seed} terminé", flush=True)
    rows = []
    for (seed, tag), (yy, pp) in store.items():
        yy, pp = np.array(yy), np.array(pp); prev = yy.mean()
        b = brier_score_loss(yy, pp); b0 = brier_score_loss(yy, np.full_like(pp, prev))
        rows.append(dict(dataset=name, seed=seed, method=tag, brier=b, bss=1 - b / b0, ece=ece_score(yy, pp),
                         ece_q=ece_score(yy, pp, strategy="quantile"), mean_p=pp.mean(), prev=prev,
                         auprc=average_precision_score(yy, pp)))
    return pd.DataFrame(rows), pd.DataFrame(fold_brier)

rng_b = np.random.RandomState(0)
def report_cal(R, F, name):
    D = R[R.dataset == name]; G = F[F.dataset == name]
    prev = D.prev.mean()
    print("\n" + "=" * 100)
    print(f"  {name.upper()}  — prévalence {prev:.3f} | Brier du prédicteur constant (prévalence) = {prev*(1-prev):.4f}")
    print("  (± = écart-type entre seeds ; ECE : 10 classes de même largeur | ECE_q : 10 classes de même effectif ; BSS = 1 − Brier/Brier_prévalence)")
    print("=" * 100)
    print(f"   {'méthode':<26} {'Brier':>14} {'BSS':>7} {'ECE':>14} {'ECE_q':>8} {'p moyen':>8} {'AUPRC':>7}")
    for m in [x for b_ in METHODS for x in (b_, b_ + " +Platt")]:
        d = D[D.method == m]
        if len(d) == 0: continue
        print(f"   {m:<26} {d.brier.mean():.4f}±{d.brier.std(ddof=0):.4f} {d.bss.mean():>7.3f} {d.ece.mean():.4f}±{d.ece.std(ddof=0):.4f} "
              f"{d.ece_q.mean():>8.4f} {d.mean_p.mean():>8.3f} {d.auprc.mean():>7.3f}")
    # Brier apparié par pli (moyenne des seeds) : MPRF − autres
    fa = G.groupby(["method", "fold"]).brier.mean().reset_index()
    print("\n   Brier apparié par pli (moyenne des seeds) : MPRF − autre  (négatif = MPRF mieux calibré/plus précis) ; IC95% bootstrap, Wilcoxon")
    for other in ["CSRF", "OBP", "IBEP", MAIN + " +Platt"]:
        x = fa[fa.method == MAIN].sort_values("fold").brier.values; y_ = fa[fa.method == other].sort_values("fold").brier.values
        dd = x - y_; bm = [rng_b.choice(dd, len(dd)).mean() for _ in range(2000)]
        try: p = 1.0 if np.all(dd == 0) else wilcoxon(x, y_)[1]
        except Exception: p = np.nan
        print(f"   MPRF − {other:<22} Δ={dd.mean():+.4f} [{np.percentile(bm,2.5):+.4f},{np.percentile(bm,97.5):+.4f}]  p brut={p:.4f}")

t0 = time.time(); allR, allF = [], []
for name in DATASETS:
    ld = load(name)
    if ld is None: continue
    X, y = ld; c = np.bincount(y)
    print(f"\n### {name} : {X.shape} | classes {c.tolist()} | IR={c[0]/c[1]:.1f}:1")
    R, F = run_cal(name, X, y); allR.append(R); allF.append(F)
    report_cal(pd.concat(allR), pd.concat(allF), name)
    pd.concat(allR).to_csv(os.path.join(OUT_DIR, "mprf_calibration_raw.csv"), index=False)
print(f"\nTerminé en {(time.time()-t0)/60:.1f} min — résultats : mprf_calibration_raw.csv")
print("Lecture : Brier/ECE plus bas = mieux calibré. p moyen à comparer à la prévalence (biais de calibration global).")
print("Platt est ajusté sur la partition de validation (peu de cas minoritaires sur Hépatite/Cervical : résultat instable).")



### cervical : (858, 35) | classes [803, 55] | IR=14.6:1
   cervical: seed 0 terminé
   cervical: seed 1 terminé
   cervical: seed 2 terminé
   cervical: seed 3 terminé
   cervical: seed 4 terminé

  CERVICAL  — prévalence 0.064 | Brier du prédicteur constant (prévalence) = 0.0600
  (± = écart-type entre seeds ; ECE : 10 classes de même largeur | ECE_q : 10 classes de même effectif ; BSS = 1 − Brier/Brier_prévalence)
   méthode                             Brier     BSS            ECE    ECE_q  p moyen   AUPRC
   CSRF                       0.0331±0.0007   0.448 0.0193±0.0029   0.0216    0.058   0.600
   CSRF +Platt                0.0371±0.0011   0.381 0.0243±0.0034   0.0130    0.065   0.580
   OBP                        0.0369±0.0038   0.384 0.0230±0.0049   0.0136    0.061   0.550
   OBP +Platt                 0.0412±0.0036   0.313 0.0287±0.0061   0.0148    0.061   0.549
   IBEP                       0.0394±0.0027   0.343 0.0243±0.0036   0.0178    0.059   0.491
   IBEP +Platt          